In [2]:
# ==========================================
# 1. PREPARAÇÃO DOS DADOS
# ==========================================

import pandas as pd

df = pd.read_csv('data/dataset.csv')

textos = df['text'].tolist()
labels = df['label'].tolist()

In [3]:
import torch
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification, 
    AutoConfig,
    Trainer, 
    TrainingArguments,
    EarlyStoppingCallback
)
from datasets import Dataset
import os
os.environ["WANDB_DISABLED"] = "true"

# ==========================================
# FUNÇÃO DE MÉTRICAS
# ==========================================
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    
    # average='weighted' é ideal para datasets pequenos que possam ter classes desbalanceadas
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, average='weighted', zero_division=0
    )
    acc = accuracy_score(labels, predictions)
    
    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

# ==========================================
# 2. CONFIGURAÇÃO DO MODELO E K-FOLD
# ==========================================
model_name = "answerdotai/ModernBERT-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

resultados_finais = []

# --- VARIÁVEIS PARA RASTREAR O MELHOR MODELO GLOBAL ---
melhor_f1_global = 0.0
melhor_fold_global = 0
caminho_exportacao = './modelo_subm3'

for fold, (train_idx, val_idx) in enumerate(skf.split(textos, labels)):
    print(f"\n{'='*50}")
    print(f"A INICIAR FOLD {fold + 1}/3")
    print(f"{'='*50}")
    
    train_texts = [textos[i] for i in train_idx]
    train_labels = [labels[i] for i in train_idx]
    val_texts = [textos[i] for i in val_idx]
    val_labels = [labels[i] for i in val_idx]
    
    train_dataset = Dataset.from_dict({'text': train_texts, 'label': train_labels})
    val_dataset = Dataset.from_dict({'text': val_texts, 'label': val_labels})
    
    def tokenize_function(examples):
        return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=512)
    
    tokenized_train = train_dataset.map(tokenize_function, batched=True)
    tokenized_val = val_dataset.map(tokenize_function, batched=True)
    
    # ==========================================
    # 3. CONFIGURAÇÃO DO MODELO
    # ==========================================
    config = AutoConfig.from_pretrained(
        model_name, 
        num_labels=5, 
        classifier_dropout=0.3 
    )
    
    model = AutoModelForSequenceClassification.from_pretrained(model_name, config=config)
    
    # ==========================================
    # 4. TRAINING ARGUMENTS
    # ==========================================
    training_args = TrainingArguments(
        output_dir=f'./resultados_fold_{fold+1}',
        eval_strategy="epoch",        
        save_strategy="epoch",        
        logging_strategy="epoch",          
        load_best_model_at_end=True,  
        metric_for_best_model="eval_f1",   
        greater_is_better=True,            
        learning_rate=2e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        num_train_epochs=3,                
        warmup_ratio=0.1,                  
        weight_decay=0.01,
        fp16=torch.cuda.is_available(),    
        logging_dir=f'./logs_fold_{fold+1}',
        report_to="none"                   
    )
    
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_train,
        eval_dataset=tokenized_val,
        compute_metrics=compute_metrics,   
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)] 
    )
    
    # Treinar
    trainer.train()
    
    # ==========================================
    # 5. EXTRAIR MÉTRICAS
    # ==========================================
    # O trainer.state.best_metric guarda o melhor F1-Score que encontrou durante o treino
    if trainer.state.best_metric is not None:
        f1_do_fold = float(trainer.state.best_metric)
    else:
        # Prevenção caso o treino pare na 1ª época
        f1_do_fold = 0.0 
        
    print(f"\nTreino do Fold {fold + 1} concluído!")
    print(f"F1-Score atingido: {f1_do_fold:.4f}")
    
    resultados_finais.append(f1_do_fold)
    
    # ==========================================
    # 6. EXPORTAR SE FOR O MELHOR MODELO
    # ==========================================
    if f1_do_fold > melhor_f1_global:
        melhor_f1_global = f1_do_fold
        melhor_fold_global = fold + 1
        
        print(f"Este é o melhor modelo até agora. A exportar...")
        # Guardar o modelo e o tokenizer na pasta final
        model.save_pretrained(caminho_exportacao)
        tokenizer.save_pretrained(caminho_exportacao)

# Calcular a média final
f1_medio = np.mean(resultados_finais)

print(f"\n{'='*50}")
print(f"TREINO TOTAL CONCLUÍDO!")
print(f"F1-Score Médio dos 3 Folds: {f1_medio:.4f} ({(f1_medio*100):.2f}%)")
print(f"O melhor modelo absoluto foi o do Fold {melhor_fold_global} com F1={melhor_f1_global:.4f}")
print(f"Está guardado na pasta: {caminho_exportacao}")
print(f"{'='*50}")


A INICIAR FOLD 1/3


Loading weights: 100%|██████████| 136/136 [00:00<00:00, 6623.23it/s]
ModernBertForSequenceClassification LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
classifier.bias   | MISSING    | 
classifier.weight | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.632973,0.209792,0.921426,0.921096,0.926905,0.921426
2,0.142510,0.082892,0.977501,0.977354,0.977537,0.977501
3,0.029851,0.069929,0.982001,0.981997,0.982022,0.982001


Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.27s/it]



Treino do Fold 1 concluído!
F1-Score atingido: 0.9820
Este é o melhor modelo até agora. A exportar...


...


Treino do Fold 2 concluído!
F1-Score atingido: 0.9770

A INICIAR FOLD 3/3


Loading weights: 100%|██████████| 136/136 [00:00<00:00, 3601.83it/s]
ModernBertForSequenceClassification LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
classifier.bias   | MISSING    | 
classifier.weight | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.567395,0.300179,0.899931,0.899898,0.925057,0.899931
2,0.138965,0.095718,0.973684,0.973660,0.974736,0.973684
3,0.039467,0.072416,0.981302,0.981259,0.981292,0.981302


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.06it/s]



Treino do Fold 3 concluído!
F1-Score atingido: 0.9813

TREINO TOTAL CONCLUÍDO!
F1-Score Médio dos 3 Folds: 0.9801 (98.01%)
O melhor modelo absoluto foi o do Fold 1 com F1=0.9820
Está guardado na pasta: ./modelo_subm3
